# AI-based Crowd Density Detection with Multi-Factor Switching

## Quick Start with GPU
1. **Runtime Setup**: Go to **Runtime -> Change runtime type** and select **T4 GPU** (or any available GPU).
2. **Run Cells**: Execute the cells below sequentially.
3. **Upload Video**: After cloning, upload your `video.mp4` to the 'Files' tab on the left.

## New Features ✨
- **Multi-Factor Model Switching**: Intelligently switches between YOLO and CSRNet based on:
  - YOLO count threshold (≥ 30)
  - Bounding box overlap (≥ 15%)
  - Crowd coverage area (≥ 25%)
  - Sudden count drops (occlusion detection)
- **GPU Acceleration**: 10-50x faster on Colab GPU
- **Enhanced Logging**: CSV includes switching reasons

In [ ]:
# Check if GPU is available
!nvidia-smi

In [ ]:
# Clone the repository
!git clone https://github.com/PoojaSadashivBansode/AI-based_Crowd_Density_Detection.git

# Change directory to the cloned repo
import os
os.chdir('AI-based_Crowd_Density_Detection')
print('Current Working Directory:', os.getcwd())

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt

# Download CSRNet weights automatically from Hugging Face
print('Downloading CSRNet weights...')
!wget -q https://huggingface.co/muasifk/CSRNet/resolve/main/CSRNet.pth -O csrnet_weights.pth
print('✅ Dependencies and weights ready!')

## Upload Your Video
Upload your `video.mp4` using the Files sidebar (📁 icon on the left), then run the next cell.

In [ ]:
# Check for video and process with multi-factor switching
import os
import shutil

video_filename = 'video.mp4'

# Check if video is in the parent directory and move it here
if not os.path.exists(video_filename) and os.path.exists(f'../{video_filename}'):
    print(f"Found {video_filename} in parent directory. Moving it...")
    shutil.move(f'../{video_filename}', video_filename)

if not os.path.exists(video_filename):
    print(f"❌ ERROR: {video_filename} not found! Please upload it to the Files sidebar.")
else:
    print(f"✅ Found {video_filename}. Starting GPU-accelerated processing...")
    print("")
    print("🧠 Using Multi-Factor Model Switching:")
    print("   • YOLO count ≥ 30")
    print("   • Bounding box overlap ≥ 15%")
    print("   • Crowd coverage ≥ 25%")
    print("   • Sudden count drops (occlusion)")
    print("")
    # Use GPU-optimized script with multi-factor switching
    !python main.py --source video.mp4 --threshold 50 --yolo yolov8s.pt

In [ ]:
# Display the output video
from IPython.display import HTML, display
from base64 import b64encode
import os

def play_video(video_path):
    if not os.path.exists(video_path):
        print("❌ Video file not found. Did the script run successfully?")
        return
    print(f"✅ Playing {video_path}...")
    mp4 = open(video_path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    return HTML(f"""
    <video width=800 controls>
          <source src="{data_url}" type="video/mp4">
    </video>
    """)

# Play the output video
play_video('output.mp4')

In [ ]:
# View the detailed CSV log with switching reasons
import pandas as pd

if os.path.exists('crowd_data.csv'):
    df = pd.read_csv('crowd_data.csv')
    print("📊 Crowd Detection Log (First 20 frames):")
    display(df.head(20))
    print(f"\n📈 Summary Statistics:")
    print(f"   • Total Frames: {len(df)}")
    print(f"   • YOLO Frames: {(df['Mode'] == 'YOLO').sum()}")
    print(f"   • CSRNet Frames: {(df['Mode'] == 'CSRNet').sum()}")
    print(f"   • Alert Frames: {(df['Alert'] == 'YES').sum()}")
    print(f"   • Average Count: {df['Count'].mean():.1f}")
    print(f"   • Peak Count: {df['Count'].max()}")
else:
    print("❌ CSV log not found.")

## Run Streamlit Dashboard in Colab (Optional)
⚠️ **Note**: The dashboard works but requires tunneling. For best experience, use the script above or run locally.

The dashboard now includes:
- 🔔 Alarm sound when threshold exceeded
- 🧠 Multi-factor model switching visualization
- 📊 Real-time switching reason display

In [ ]:
# RUN DASHBOARD IN COLAB (Optional)
print("Installing dependencies for tunnel...")
!npm install -q localtunnel

print("Getting public IP...")
!wget -q -O - ipv4.icanhazip.com
print("^ COPY THIS IP ADDRESS ^")

print("Starting Streamlit dashboard...")
!streamlit run app.py &>/content/logs.txt &

print("Opening Tunnel... Click the link below and paste the IP address!")
!npx localtunnel --port 8501